<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-35.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastapi uvicorn pytest faiss-cpu scikit-learn redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 28.4 MB/s eta 0:00:00


In [2]:
import time
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("day35")


class CircuitBreaker:
    def __init__(self, failure_threshold=3, recovery_timeout=60):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failure_count = 0
        self.last_failure_time = None
        self.state = "CLOSED"

    def call(self, function, *args, **kwargs):
        if self.state == "OPEN":
            elapsed = time.time() - self.last_failure_time

            if elapsed < self.recovery_timeout:
                logger.warning("Circuit breaker is OPEN")
                return "Service temporarily unavailable"

            self.state = "HALF_OPEN"
            logger.info("Circuit breaker entering HALF_OPEN state")

        try:
            result = function(*args, **kwargs)

            self.failure_count = 0
            self.state = "CLOSED"

            logger.info("AI call successful")
            return result

        except Exception as e:
            self.failure_count += 1
            self.last_failure_time = time.time()

            logger.error(f"AI call failed: {e}")

            if self.failure_count >= self.failure_threshold:
                self.state = "OPEN"
                logger.warning("Circuit breaker OPENED")

            raise


class MockAIService:
    def generate(self, prompt):
        logger.debug(f"Processing prompt: {prompt}")

        return f"Mock AI response for: {prompt}"


ai_service = MockAIService()
circuit_breaker = CircuitBreaker()

response = circuit_breaker.call(
    ai_service.generate,
    "What is Retrieval Augmented Generation?"
)

print(response)
print("Circuit state:", circuit_breaker.state)

Mock AI response for: What is Retrieval Augmented Generation?
Circuit state: CLOSED


In [3]:
class FailingAIService:
    def generate(self, prompt):
        raise Exception("Simulated AI service failure")


failing_service = FailingAIService()

for i in range(4):
    try:
        result = circuit_breaker.call(
            failing_service.generate,
            "Test request"
        )
        print(f"Call {i + 1}: {result}")

    except Exception as e:
        print(f"Call {i + 1}: Failed - {e}")

print("\nFinal circuit state:", circuit_breaker.state)
print("Failure count:", circuit_breaker.failure_count)

ERROR:day35:AI call failed: Simulated AI service failure
ERROR:day35:AI call failed: Simulated AI service failure
ERROR:day35:AI call failed: Simulated AI service failure


Call 1: Failed - Simulated AI service failure
Call 2: Failed - Simulated AI service failure
Call 3: Failed - Simulated AI service failure
Call 4: Service temporarily unavailable

Final circuit state: OPEN
Failure count: 3


In [4]:
import numpy as np
import faiss

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


documents = [
    "Retrieval Augmented Generation combines retrieval with language generation.",
    "FAISS is used for efficient similarity search over vector embeddings.",
    "TF-IDF retrieves documents using keyword-based similarity.",
    "FastAPI is a Python framework for building APIs.",
    "Redis is an in-memory data store commonly used for caching."
]


# Simple local embeddings for demonstration
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

dimension = tfidf_matrix.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

vectors = tfidf_matrix.toarray().astype("float32")
faiss_index.add(vectors)


def faiss_search(query, k=2):
    query_vector = vectorizer.transform([query]).toarray().astype("float32")

    distances, indices = faiss_index.search(query_vector, k)

    return [documents[i] for i in indices[0]]


def tfidf_search(query, k=2):
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    top_indices = similarities.argsort()[-k:][::-1]

    return [documents[i] for i in top_indices]


def retrieve(query, k=2):
    try:
        logger.info("Trying FAISS retrieval")
        results = faiss_search(query, k)
        logger.info("FAISS retrieval successful")
        return results, "FAISS"

    except Exception as e:
        logger.warning(f"FAISS failed: {e}")
        logger.info("Falling back to TF-IDF retrieval")

        results = tfidf_search(query, k)
        return results, "TF-IDF"


results, method = retrieve("What is RAG?")

print("Retrieval method:", method)
print("\nRetrieved documents:")

for result in results:
    print("-", result)

Retrieval method: FAISS

Retrieved documents:
- FastAPI is a Python framework for building APIs.
- FAISS is used for efficient similarity search over vector embeddings.


In [5]:
original_faiss_search = faiss_search


def broken_faiss_search(query, k=2):
    raise Exception("Simulated FAISS failure")


faiss_search = broken_faiss_search


results, method = retrieve("What is RAG?")

print("Retrieval method:", method)

print("\nRetrieved documents:")
for result in results:
    print("-", result)


faiss_search = original_faiss_search

Retrieval method: TF-IDF

Retrieved documents:
- FastAPI is a Python framework for building APIs.
- FAISS is used for efficient similarity search over vector embeddings.


In [6]:
from fastapi import FastAPI
from datetime import datetime

app = FastAPI(title="Day 35 Production RAG API")


def check_faiss():
    try:
        test_vector = np.zeros((1, dimension), dtype="float32")
        faiss_index.search(test_vector, 1)

        return {
            "status": "healthy",
            "message": "FAISS index is loaded and queryable"
        }

    except Exception as e:
        return {
            "status": "unhealthy",
            "message": str(e)
        }


def check_redis():
    try:
        import redis

        client = redis.Redis(
            host="localhost",
            port=6379,
            socket_connect_timeout=1
        )

        client.ping()

        return {
            "status": "healthy",
            "message": "Redis is reachable"
        }

    except Exception as e:
        return {
            "status": "unhealthy",
            "message": "Redis unavailable"
        }


def check_ai():
    try:
        response = circuit_breaker.call(
            ai_service.generate,
            "health check"
        )

        return {
            "status": "healthy",
            "message": "AI service responded"
        }

    except Exception as e:
        return {
            "status": "unhealthy",
            "message": str(e)
        }


@app.get("/health")
def health_check():

    faiss_status = check_faiss()
    redis_status = check_redis()
    ai_status = check_ai()

    overall_status = "healthy"

    if any(
        check["status"] == "unhealthy"
        for check in [
            faiss_status,
            redis_status,
            ai_status
        ]
    ):
        overall_status = "degraded"

    return {
        "status": overall_status,
        "timestamp": datetime.utcnow().isoformat(),
        "checks": {
            "faiss": faiss_status,
            "redis": redis_status,
            "ai_service": ai_status
        }
    }

In [7]:
!apt-get update -qq
!apt-get install -y -qq redis-server
!redis-server --daemonize yes

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../libjemalloc2_5.3.0-2build1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.3.0-2build1) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../liblzf1_3.6-4_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-4) ...
Selecting previously unselected package redis-tools.
Preparing to unpack .../redis-tools_5%3a7.0.15-1ubuntu0.24.04.4_amd64.deb ...
Unpacking redis-tools (5:7.0.15-1ubuntu0.24.04.4) ...
Selecting previously unselected package redis-server.
Preparing to unpack .../redis-server_5%3a7.0.15-1ubuntu0.24.04.4_amd64.deb ...
Unpacking redis-server (5:7.0.15-1ubuntu0.24.04.4) ...
Setting up libjemalloc2:amd64 (5.3.0-2build1) ..

In [8]:
import redis

redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

print("Redis ping:", redis_client.ping())

Redis ping: True


In [9]:
health_result = health_check()

print(health_result)

{'status': 'healthy', 'timestamp': '2026-09-17T13:08:15.886714', 'checks': {'faiss': {'status': 'healthy', 'message': 'FAISS index is loaded and queryable'}, 'redis': {'status': 'healthy', 'message': 'Redis is reachable'}, 'ai_service': {'status': 'healthy', 'message': 'AI service responded'}}}


/tmp/ipykernel_2465/551513106.py:88: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


In [10]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


In [11]:
import requests

response = requests.get("http://127.0.0.1:8000/health")

print("Status code:", response.status_code)
print("\nHealth response:")
print(response.json())

INFO:     127.0.0.1:39886 - "GET /health HTTP/1.1" 200 OK
Status code: 200

Health response:
{'status': 'healthy', 'timestamp': '2026-09-17T13:08:47.320289', 'checks': {'faiss': {'status': 'healthy', 'message': 'FAISS index is loaded and queryable'}, 'redis': {'status': 'healthy', 'message': 'Redis is reachable'}, 'ai_service': {'status': 'healthy', 'message': 'AI service responded'}}}


/tmp/ipykernel_2465/551513106.py:88: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


In [12]:
import logging

logger = logging.getLogger("day35.production")


def configure_logging(level=logging.INFO):
    logger.setLevel(level)

    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            "%(asctime)s - %(levelname)s - %(message)s"
        )

        handler.setFormatter(formatter)
        logger.addHandler(handler)

    for handler in logger.handlers:
        handler.setLevel(level)


def simulate_production_logging():
    logger.debug("Debug: detailed internal information")
    logger.info("Info: system operating normally")
    logger.warning("Warning: service response is slow")
    logger.error("Error: service request failed")


configure_logging(logging.DEBUG)

print("Logging at DEBUG level:")
simulate_production_logging()

2026-09-17 13:09:02,551 - DEBUG - Debug: detailed internal information
DEBUG:day35.production:Debug: detailed internal information
2026-09-17 13:09:02,572 - INFO - Info: system operating normally
INFO:day35.production:Info: system operating normally
2026-09-17 13:09:02,582 - WARNING - Warning: service response is slow
2026-09-17 13:09:02,591 - ERROR - Error: service request failed
ERROR:day35.production:Error: service request failed


Logging at DEBUG level:


In [13]:
configure_logging(logging.WARNING)

print("\nLogging at WARNING level:")
simulate_production_logging()

2026-09-17 13:09:10,896 - WARNING - Warning: service response is slow
2026-09-17 13:09:10,913 - ERROR - Error: service request failed
ERROR:day35.production:Error: service request failed



Logging at WARNING level:


In [14]:
%%writefile test_day35.py

import pytest
from fastapi.testclient import TestClient

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from day35_app import (
    app,
    retrieve,
    faiss_search,
    ai_service,
    CircuitBreaker
)


client = TestClient(app)


def test_health_endpoint():
    response = client.get("/health")

    assert response.status_code == 200

    data = response.json()

    assert "status" in data
    assert "checks" in data
    assert "faiss" in data["checks"]
    assert "redis" in data["checks"]
    assert "ai_service" in data["checks"]


def test_faiss_retrieval():
    results, method = retrieve("What is RAG?")

    assert method == "FAISS"
    assert len(results) > 0


def test_tfidf_fallback(monkeypatch):

    def failing_search(query, k=2):
        raise Exception("FAISS failure")

    monkeypatch.setattr(
        "day35_app.faiss_search",
        failing_search
    )

    results, method = retrieve("What is RAG?")

    assert method == "TF-IDF"
    assert len(results) > 0


def test_circuit_breaker():

    breaker = CircuitBreaker(
        failure_threshold=3,
        recovery_timeout=60
    )

    def failing_service():
        raise Exception("Simulated failure")

    for _ in range(3):
        with pytest.raises(Exception):
            breaker.call(failing_service)

    assert breaker.state == "OPEN"

    result = breaker.call(failing_service)

    assert result == "Service temporarily unavailable"


def test_end_to_end_pipeline():

    query = "What is Retrieval Augmented Generation?"

    documents, method = retrieve(query)

    assert len(documents) > 0

    prompt = (
        "Answer the question using the retrieved documents: "
        + " ".join(documents)
        + "\nQuestion: "
        + query
    )

    response = ai_service.generate(prompt)

    assert response is not None
    assert len(response) > 0

Writing test_day35.py


In [15]:
%%writefile day35_app.py

import time
import logging
import numpy as np
import faiss

from fastapi import FastAPI
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("day35")


class CircuitBreaker:

    def __init__(self, failure_threshold=3, recovery_timeout=60):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failure_count = 0
        self.last_failure_time = None
        self.state = "CLOSED"

    def call(self, function, *args, **kwargs):

        if self.state == "OPEN":

            elapsed = time.time() - self.last_failure_time

            if elapsed < self.recovery_timeout:
                logger.warning("Circuit breaker is OPEN")
                return "Service temporarily unavailable"

            self.state = "HALF_OPEN"

        try:
            result = function(*args, **kwargs)

            self.failure_count = 0
            self.state = "CLOSED"

            return result

        except Exception as e:

            self.failure_count += 1
            self.last_failure_time = time.time()

            logger.error(f"AI call failed: {e}")

            if self.failure_count >= self.failure_threshold:
                self.state = "OPEN"

            raise


class MockAIService:

    def generate(self, prompt):
        logger.debug("Generating mock AI response")
        return f"Mock AI response for: {prompt}"


ai_service = MockAIService()
circuit_breaker = CircuitBreaker()


documents = [
    "Retrieval Augmented Generation combines retrieval with language generation.",
    "FAISS is used for efficient similarity search over vector embeddings.",
    "TF-IDF retrieves documents using keyword-based similarity.",
    "FastAPI is a Python framework for building APIs.",
    "Redis is an in-memory data store commonly used for caching."
]


vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

dimension = tfidf_matrix.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

vectors = tfidf_matrix.toarray().astype("float32")
faiss_index.add(vectors)


def faiss_search(query, k=2):

    query_vector = vectorizer.transform(
        [query]
    ).toarray().astype("float32")

    distances, indices = faiss_index.search(
        query_vector,
        k
    )

    return [documents[i] for i in indices[0]]


def tfidf_search(query, k=2):

    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    top_indices = similarities.argsort()[-k:][::-1]

    return [documents[i] for i in top_indices]


def retrieve(query, k=2):

    try:

        logger.info("Trying FAISS retrieval")

        results = faiss_search(query, k)

        return results, "FAISS"

    except Exception as e:

        logger.warning(f"FAISS failed: {e}")
        logger.info("Using TF-IDF fallback")

        results = tfidf_search(query, k)

        return results, "TF-IDF"


app = FastAPI(
    title="Day 35 Production RAG API"
)


def check_faiss():

    try:

        test_vector = np.zeros(
            (1, dimension),
            dtype="float32"
        )

        faiss_index.search(
            test_vector,
            1
        )

        return {
            "status": "healthy",
            "message": "FAISS index is loaded and queryable"
        }

    except Exception as e:

        return {
            "status": "unhealthy",
            "message": str(e)
        }


def check_redis():

    try:

        import redis

        client = redis.Redis(
            host="localhost",
            port=6379,
            socket_connect_timeout=1
        )

        client.ping()

        return {
            "status": "healthy",
            "message": "Redis is reachable"
        }

    except Exception:

        return {
            "status": "unhealthy",
            "message": "Redis unavailable"
        }


def check_ai():

    try:

        response = circuit_breaker.call(
            ai_service.generate,
            "health check"
        )

        return {
            "status": "healthy",
            "message": "AI service responded"
        }

    except Exception as e:

        return {
            "status": "unhealthy",
            "message": str(e)
        }


@app.get("/health")
def health_check():

    faiss_status = check_faiss()
    redis_status = check_redis()
    ai_status = check_ai()

    checks = [
        faiss_status,
        redis_status,
        ai_status
    ]

    overall_status = "healthy"

    if any(
        check["status"] == "unhealthy"
        for check in checks
    ):
        overall_status = "degraded"

    return {
        "status": overall_status,
        "timestamp": datetime.utcnow().isoformat(),
        "checks": {
            "faiss": faiss_status,
            "redis": redis_status,
            "ai_service": ai_status
        }
    }

Writing day35_app.py


In [16]:
!pytest -q test_day35.py

.....                                                                    [100%]
5 passed in 2.86s


In [17]:
from day35_app import retrieve, ai_service

evaluation_cases = [
    {
        "question": "What is Retrieval Augmented Generation?",
        "expected": "Retrieval Augmented Generation combines retrieval with language generation."
    },
    {
        "question": "What is FAISS?",
        "expected": "FAISS is used for efficient similarity search over vector embeddings."
    },
    {
        "question": "What is TF-IDF?",
        "expected": "TF-IDF retrieves documents using keyword-based similarity."
    },
    {
        "question": "What is FastAPI?",
        "expected": "FastAPI is a Python framework for building APIs."
    },
    {
        "question": "What is Redis?",
        "expected": "Redis is an in-memory data store commonly used for caching."
    }
]


results = []

for case in evaluation_cases:

    retrieved_docs, method = retrieve(case["question"])

    answer = ai_service.generate(
        "Question: "
        + case["question"]
        + "\nContext: "
        + " ".join(retrieved_docs)
    )

    context_match = any(
        case["expected"] in doc
        for doc in retrieved_docs
    )

    results.append({
        "question": case["question"],
        "retrieval_method": method,
        "context_match": context_match,
        "answer_generated": bool(answer)
    })


retrieval_score = sum(
    result["context_match"]
    for result in results
) / len(results) * 100

generation_score = sum(
    result["answer_generated"]
    for result in results
) / len(results) * 100

overall_score = (
    retrieval_score + generation_score
) / 2


print("DAY 35 PRODUCTION BASELINE")
print("-" * 35)
print(f"Retrieval score : {retrieval_score:.1f}%")
print(f"Generation score: {generation_score:.1f}%")
print(f"Overall baseline: {overall_score:.1f}%")

DAY 35 PRODUCTION BASELINE
-----------------------------------
Retrieval score : 100.0%
Generation score: 100.0%
Overall baseline: 100.0%


In [18]:
deployment_checklist = [
    "1. Circuit breaker opens after 3 consecutive AI service failures and blocks requests for 60 seconds.",
    "2. GET /health returns structured status for FAISS, Redis, and AI service.",
    "3. FAISS index is loaded successfully and responds to a test query.",
    "4. Redis is reachable and responds successfully to PING.",
    "5. TF-IDF fallback activates automatically when FAISS retrieval fails.",
    "6. All 5 integration tests pass without using a real AI API.",
    "7. DEBUG, INFO, WARNING, and ERROR logging levels are configured and verified.",
    "8. Production evaluation baseline has been recorded and stored with the release.",
    "9. No API keys, passwords, or other secrets are hard-coded in the source code.",
    "10. The application starts successfully and the /health endpoint reports the expected production status."
]

print("DAY 35 PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 50)

for item in deployment_checklist:
    print(item)

DAY 35 PRODUCTION DEPLOYMENT CHECKLIST
1. Circuit breaker opens after 3 consecutive AI service failures and blocks requests for 60 seconds.
2. GET /health returns structured status for FAISS, Redis, and AI service.
3. FAISS index is loaded successfully and responds to a test query.
4. Redis is reachable and responds successfully to PING.
5. TF-IDF fallback activates automatically when FAISS retrieval fails.
6. All 5 integration tests pass without using a real AI API.
7. DEBUG, INFO, WARNING, and ERROR logging levels are configured and verified.
8. Production evaluation baseline has been recorded and stored with the release.
9. No API keys, passwords, or other secrets are hard-coded in the source code.
10. The application starts successfully and the /health endpoint reports the expected production status.
